CHIẾN LƯỢC: BUY THE DIP M15
Khung thời gian: 15 Phút (M15).

Vào lệnh (Entry): MA9 > MA200 VÀ RSI < 35. (Mua khi đang uptrend nhưng giá điều chỉnh ngắn hạn).

Dừng lỗ (Stop Loss): Đặt tại MA200 - (1.5 x ATR). (Cắt lỗ dưới hỗ trợ xu hướng, có khoảng đệm tránh râu nến).

Chốt lời (Take Profit): Thoát lệnh khi MA9 > MA50 VÀ Giá > MA9. (Chốt khi cấu trúc tăng giá ngắn hạn xác nhận quay trở lại).

In [ ]:
# Install required libraries
!pip install pandas pandas-ta requests plotly -q

import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# ==========================================
# GLOBAL CONFIGURATION
# ==========================================
CONFIG = {
    'SYMBOLS': ['BTCUSDT', 'ETHUSDT'],  # Test trên BTC và ETH
    'INTERVAL': '15m',                  # Khung thời gian M15
    'START_DATE': '01/01/2023',         # Dữ liệu từ đầu năm 2023
    'END_DATE': '01/01/2024',           # Đến đầu năm 2024
    'INITIAL_CAPITAL': 10000.0,
    'POSITION_SIZE_PCT': 0.10,          # Vào lệnh 10% vốn mỗi trade
    'FEE_RATE': 0.0004,                 # Phí giao dịch (Binance VIP 0 maker/taker)
    'SLIPPAGE': 0.0002                  # Trượt giá
}

# ==========================================
# MODULE 1: DATA HANDLER
# ==========================================
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        req_symbol = self.symbol

        print(f"Checking connection for {req_symbol}...")
        try:
            test = requests.get(f"{base_url}/api/v3/time", timeout=5)
            if test.status_code == 451:
                base_url = "https://api.binance.us"
        except Exception:
            base_url = "https://api.binance.us"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)

        if self.end_date == datetime.now().strftime("%d/%m/%Y"):
            end_time = int(time.time() * 1000)
        else:
            end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
            end_time = int(end_dt.timestamp() * 1000)

        all_klines =[]
        print(f"Downloading {req_symbol} ({self.interval})...")

        while start_time < end_time:
            params = {"symbol": req_symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except Exception: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df.astype(float)
        print(f"Total: {len(df)} candles fetched.")
        return df

# ==========================================
# MODULE 2: STRATEGY LOGIC (BUY THE DIP M15)
# ==========================================
class BuyTheDipStrategy:
    def __init__(self):
        # Thông số chỉ báo
        self.ma_9 = 9
        self.ma_50 = 50
        self.ma_200 = 200
        self.rsi_period = 14
        self.rsi_entry = 35
        self.atr_period = 14
        self.atr_multiplier = 1.5

    def calculate_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        # Sử dụng SMA (Simple Moving Average) cho các đường MA
        df['MA_9'] = ta.sma(df['Close'], length=self.ma_9)
        df['MA_50'] = ta.sma(df['Close'], length=self.ma_50)
        df['MA_200'] = ta.sma(df['Close'], length=self.ma_200)

        # RSI & ATR
        df['RSI'] = ta.rsi(df['Close'], length=self.rsi_period)
        df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'], length=self.atr_period)

        df.dropna(inplace=True)
        return df

# ==========================================
# MODULE 3: BACKTEST ENGINE
# ==========================================
class BacktestEngine:
    def __init__(self, df, strategy, config):
        self.df = strategy.calculate_indicators(df.copy())
        self.strategy = strategy
        self.config = config

        self.capital = config['INITIAL_CAPITAL']
        self.invested_amount = 0.0
        self.position = 0
        self.entry_price = 0.0
        self.stop_loss_price = 0.0 # Thêm biến lưu giá Stop Loss cố định lúc vào lệnh
        self.trades =[]

        self.equity_curve = np.zeros(len(self.df))

    def run(self):
        # Trích xuất dữ liệu mảng để tối ưu tốc độ xử lý
        O = self.df['Open'].values
        H = self.df['High'].values
        L = self.df['Low'].values
        C = self.df['Close'].values
        times = self.df.index

        ma9 = self.df['MA_9'].values
        ma50 = self.df['MA_50'].values
        ma200 = self.df['MA_200'].values
        rsi = self.df['RSI'].values
        atr = self.df['ATR'].values

        fee = self.config['FEE_RATE']
        slip = self.config['SLIPPAGE']
        pos_size_pct = self.config['POSITION_SIZE_PCT']

        s = self.strategy

        self.equity_curve[0] = self.capital
        self.equity_curve[1] = self.capital

        for i in range(2, len(self.df)):

            # ---------------------------
            # EXIT LOGIC (Đóng Lệnh)
            # ---------------------------
            if self.position == 1:
                exit_signal = False
                reason = ""
                real_exit = 0.0

                # 1. Stop Loss Logic (Kiểm tra xem nến trước đó có quét qua SL không)
                if L[i-1] <= self.stop_loss_price:
                    exit_signal = True
                    reason = "Stop Loss"

                    real_exit = self.stop_loss_price * (1 - slip)

                # 2. Take Profit Logic: MA9 > MA50 VÀ Giá Đóng Cửa > MA9
                elif (ma9[i-1] > ma50[i-1]) and (C[i-1] > ma9[i-1]):
                    exit_signal = True
                    reason = "Take Profit (Trend Resumed)"
                    real_exit = O[i] * (1 - slip)

                # Thực thi đóng lệnh
                if exit_signal:
                    pnl_pct = (real_exit - self.entry_price) / self.entry_price
                    net_pnl_pct = pnl_pct - (fee * 2)
                    pnl_amt = self.invested_amount * net_pnl_pct
                    self.capital += pnl_amt

                    self.trades.append({
                        'Date': times[i], 'Type': 'Long',
                        'Reason': reason,
                        'Entry': self.entry_price, 'Exit': real_exit,
                        'PnL_Amt': pnl_amt, 'Equity': self.capital
                    })

                    # Reset trạng thái
                    self.position = 0
                    self.invested_amount = 0.0
                    self.stop_loss_price = 0.0

            # ---------------------------
            # ENTRY LOGIC (Vào Lệnh Mua)
            # ---------------------------
            if self.position == 0:
                # Điều kiện: MA9 > MA200 (Uptrend) VÀ RSI < 35 (Quá bán ngắn hạn)
                cond_uptrend = ma9[i-1] > ma200[i-1]
                cond_dip = rsi[i-1] < s.rsi_entry

                if cond_uptrend and cond_dip:
                    self.position = 1
                    self.entry_price = O[i] * (1 + slip)

                    # Xác định mức Stop Loss theo công thức: MA200 - (1.5 x ATR)
                    self.stop_loss_price = ma200[i-1] - (s.atr_multiplier * atr[i-1])

                    self.invested_amount = self.capital * pos_size_pct

            # ---------------------------
            # CALCULATE UNREALIZED PNL
            # ---------------------------
            current_equity = self.capital
            if self.position == 1:
                current_value = self.invested_amount * (C[i] / self.entry_price)
                unrealized_pnl = current_value - self.invested_amount
                current_equity += unrealized_pnl

            self.equity_curve[i] = current_equity

        return pd.Series(self.equity_curve, index=self.df.index), pd.DataFrame(self.trades)

# ==========================================
# MODULE 4: PERFORMANCE EVALUATOR
# ==========================================
class Evaluator:
    @staticmethod
    def generate_report(equity, trades_df, initial_capital, symbol, interval, pos_size_pct):
        if trades_df.empty:
            print(f"No trades executed for {symbol}!")
            return

        final_equity = equity.iloc[-1]
        net_pnl = final_equity - initial_capital
        total_return = net_pnl / initial_capital

        rolling_max = equity.cummax()
        drawdown = (equity - rolling_max) / rolling_max
        max_dd = drawdown.min()

        wins = trades_df[trades_df['PnL_Amt'] > 0]
        losses = trades_df[trades_df['PnL_Amt'] <= 0]
        winrate = len(wins) / len(trades_df) if len(trades_df) > 0 else 0
        profit_factor = wins['PnL_Amt'].sum() / abs(losses['PnL_Amt'].sum()) if len(losses) > 0 and losses['PnL_Amt'].sum() != 0 else 0

        days = (equity.index[-1] - equity.index[0]).days
        cagr = ((final_equity / initial_capital) ** (365.0 / days)) - 1 if days > 0 else 0

        returns = equity.pct_change().dropna()
        sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(365 * 24 * 4) if returns.std() != 0 else 0 # Nhân với 4 vì M15 có 4 nến/giờ

        print("")
        print(f"========== PERFORMANCE REPORT: {symbol} ({interval}) ==========")
        print(f"{'Strategy:':>20} Buy The Dip M15")
        print(f"{'Position Sizing:':>20} {pos_size_pct*100}% of Capital")
        print(f"{'Initial Capital:':>20} ${initial_capital:,.2f}")
        print(f"{'Final Equity:':>20} ${final_equity:,.2f}")
        print(f"{'Net PnL:':>20} ${net_pnl:,.2f}")
        print(f"{'Total Return:':>20} {total_return:.2%}")
        print(f"{'CAGR (Annual):':>20} {cagr:.2%}")
        print(f"{'Max Drawdown:':>20} {max_dd:.2%}")
        print(f"{'Sharpe Ratio:':>20} {sharpe_ratio:.2f}")
        print(f"{'Total Trades:':>20} {len(trades_df)}")
        print(f"{'Winrate:':>20} {winrate:.2%}")
        print(f"{'Profit Factor:':>20} {profit_factor:.2f}")
        print("====================================================\n")

        # Plotly Charts
        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.08,
            row_heights=[0.7, 0.3]
        )

        fig.add_trace(go.Scatter(
            x=equity.index, y=equity.values,
            mode="lines", name="Equity",
            line=dict(color="#2962ff", width=1.5)
        ), row=1, col=1)

        fig.add_trace(go.Scatter(
            x=drawdown.index, y=drawdown.values,
            mode="lines", name="Drawdown",
            line=dict(color="#d32f2f", width=1),
            fill='tozeroy', fillcolor='rgba(211, 47, 47, 0.2)'
        ), row=2, col=1)

        fig.update_layout(
            title=f"Portfolio Performance: {symbol} ({interval})",
            template="plotly_white",
            height=650,
            margin=dict(l=40, r=40, t=60, b=40),
            showlegend=False
        )

        fig.update_xaxes(showgrid=True, gridcolor='#e6e6e6', zeroline=False)
        fig.update_yaxes(title_text="Balance ($)", tickprefix="$", showgrid=True, gridcolor='#e6e6e6', zeroline=False, row=1, col=1)
        fig.update_yaxes(title_text="Drawdown %", tickformat=".1%", showgrid=True, gridcolor='#e6e6e6', zeroline=True, zerolinecolor='#cccccc', row=2, col=1)

        fig.show()

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    for symbol in CONFIG['SYMBOLS']:
        print(f"\n---> Starting Backtest for: {symbol} <---")
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()

        if not df_raw.empty:
            my_strategy = BuyTheDipStrategy()
            engine = BacktestEngine(df_raw, my_strategy, CONFIG)
            equity, trades = engine.run()

            Evaluator.generate_report(
                equity,
                trades,
                CONFIG['INITIAL_CAPITAL'],
                symbol,
                CONFIG['INTERVAL'],
                CONFIG['POSITION_SIZE_PCT']
            )

            # In ra 5 lệnh gần nhất để kiểm tra Logic
            if not trades.empty:
                print(f"\nTop 5 Recent Trades for {symbol}:")
                print(trades[['Date', 'Type', 'Reason', 'Entry', 'Exit', 'PnL_Amt']].tail(5).to_string(index=False))

Output hidden; open in https://colab.research.google.com to view.